# Robustness of a nearest-neighbour anomaly detector on MVTec AD

The first notebook (`mvtec_ad_comparison.ipynb`) asked *which method* separates good from defective parts best. This one asks a different question: **how much of that survives when the parts or the setup change?** A line does not see the clean, evenly lit, fixtured images of a benchmark all the time: lighting drifts, a new lot looks a little different, a part sits shifted or rotated.

The method is fixed here: the nearest-neighbour method on Wide ResNet50 features with the border margin (the code is in `helpers.py`, copied from the first notebook). Model, threshold and border margin are fitted once on good training images, then the **test images are changed** and re-scored with the unchanged model. What matters is not only the AUROC but the two numbers a line cares about: how many defects are still caught at the threshold fixed beforehand, and whether the false-reject rate stays where it was set.

Plan:
1. Setup and baseline (this section): reproduce the numbers of the first study as the reference.
2. Batch variations: simulated changes of lighting, colour and surface.
3. Shifted or rotated parts.
4. Threshold flow with a few reject samples.
5. Write-up, with limitations.

The variations are **simulated**: they are not real lot variation. The dataset is MVTec AD (CC BY-NC-SA 4.0; Bergmann et al., CVPR 2019). The dataset itself is not part of this repository, but the figures show MVTec AD images, so the same licence applies to them.

## Setup

In [ ]:
import textwrap
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_auc_score

import helpers as h
import robustness_viz as viz
import shift_check as sc
import threshold_flow as tf
from helpers import CATEGORIES

# folder with one subfolder per category: set MVTEC_AD_ROOT or edit the default in helpers.data_root()
print("data folder found" if h.data_root().exists() else "data folder not found: set MVTEC_AD_ROOT")
TARGETS = (0.05, 0.01)  # false-reject rates the threshold is aimed at, as in the first notebook

## Baseline

Same procedure as in the first notebook: the memory of good patches comes from 80% of the good training images; the other 20% are held out and fix the border margin and the threshold (the threshold is the quantile of their scores that rejects at most the target share of good parts). The test images are never used for any of this.

Two sets of numbers, to compare with the first notebook:
- image AUROC with the memory built from *all* good training images (the table of the first notebook), plain and with the border margin;
- defects caught and real false-reject rate at the 5% and 1% thresholds (the operating-point table of the first notebook, column "WRN50 + border"), with the 80% fit.

In [ ]:
REFS = {}      # category -> fitted reference, reused by the experiments below
BASE_ROWS = []

for ctg in CATEGORIES:
    t0 = time.perf_counter()
    cat = h.load_category(ctg)
    ref = h.fit_reference(cat)
    REFS[ctg] = ref

    idx = h.select(cat, "test")
    is_def = cat.labels[idx] != "good"
    s = ref.scores(cat.X[idx])
    row = {"category": ctg, "margin": ref.margin, "n_good_test": int((~is_def).sum()), "n_defective_test": int(is_def.sum())}
    for t in TARGETS:
        caught, frr = h.operating_point(ref.val_scores, s[~is_def], s[is_def], t)
        row[f"caught@{t:.0%}"], row[f"frr@{t:.0%}"] = caught, frr

    # AUROC with the memory from all good training images, as in the table of the first notebook
    maps_all = h.knn_maps(h.build_knn_memory(cat), cat.X[idx])
    row["AUROC plain"] = roc_auc_score(is_def, h.border_scores(maps_all, 0))
    row["AUROC border"] = roc_auc_score(is_def, h.border_scores(maps_all, ref.margin))
    BASE_ROWS.append(row)
    print(f"{ctg:11s} done in {time.perf_counter() - t0:.0f} s", flush=True)

baseline = pd.DataFrame(BASE_ROWS).set_index("category")
baseline.round(3)

### Check against the first study

The numbers below were read from the saved outputs of `mvtec_ad_comparison.ipynb` (AUROC table, border table, operating-point table, column "WRN50 + border"). Everything here should agree to rounding: the AUROCs to 0.001, the percentages to the whole percent.

In [ ]:
# from the first notebook: margin, AUROC plain, AUROC border, (caught, false rejects) at 5%, at 1%
FIRST = {
    "transistor": (0, 0.881, 0.881, (0.65, 0.12), (0.52, 0.10)),
    "pill":       (0, 0.853, 0.853, (0.39, 0.04), (0.28, 0.00)),
    "metal_nut":  (0, 0.958, 0.958, (0.91, 0.05), (0.73, 0.00)),
    "leather":    (2, 0.997, 1.000, (1.00, 0.06), (1.00, 0.03)),
    "carpet":     (2, 0.979, 1.000, (1.00, 0.18), (0.99, 0.11)),
    "grid":       (2, 0.772, 0.996, (1.00, 0.10), (0.98, 0.00)),
}

problems = []
for ctg, (margin, a_plain, a_border, op5, op1) in FIRST.items():
    r = baseline.loc[ctg]
    if r["margin"] != margin:
        problems.append(f"{ctg}: margin {r['margin']} instead of {margin}")
    for name, got, want in [("AUROC plain", r["AUROC plain"], a_plain), ("AUROC border", r["AUROC border"], a_border)]:
        if abs(got - want) > 0.0015:
            problems.append(f"{ctg}: {name} {got:.3f} instead of {want:.3f}")
    for t, (c, f) in zip(TARGETS, (op5, op1)):
        for name, got, want in [("caught", r[f"caught@{t:.0%}"], c), ("false rejects", r[f"frr@{t:.0%}"], f)]:
            if abs(got - want) > 0.0105:
                problems.append(f"{ctg}: {name} at {t:.0%}: {got:.0%} instead of {want:.0%}")
print("the baseline reproduces the first study" if not problems else "differences to the first study:" + chr(10) + chr(10).join(problems))

## Batch variations

A new lot, a drifting lamp, a dirty lens: the parts and the setup change a little, but the model and the threshold were fixed on the original good images. Here the **test images** are changed in twelve ways, each at three strengths, and re-scored with the unchanged model:

- lighting: darker, brighter, uneven (a brightness ramp across the image), flicker (a different gain for every image);
- contrast: lower and higher (values squeezed or stretched around the mean of the image);
- colour cast: warm (red up, blue down) and cool (the other way round);
- camera: blur and sensor noise;
- lens: dirt on the lens (the same dark spots in every image, which a model can mistake for a feature of the part);
- harmless dust or oil (random dark spots on the part that are *not* a defect: a test for overkill).

All are **simulated**. The strengths were fixed before looking at any result and are not tuned. Good and defective test images are changed in the same way. For every condition and category the cell below records:

- the image AUROC (does the ranking survive?);
- defects caught and the real false-reject rate at the 5% and 1% thresholds that were fixed on the *unchanged* held-out good images (does the threshold survive?);
- the same two numbers with a threshold set again on held-out good images changed in the same way ("recalibrated"): the difference shows how much of the damage a recalibration per lot would undo;
- the median score of the good test images, to see how far the scores move.

Random changes are seeded per image, so an image gets the same random draw at every strength.

In [ ]:
# one good image at the three strengths of every variation
demo = h.load_category("carpet")
i_demo = h.select(demo, "test", good=True)[0]
families = list(h.FAMILIES)
fig, ax = plt.subplots(len(families), 4, figsize=(8, 1.8 * len(families)))
for r, fam in enumerate(families):
    title, levels, _ = h.FAMILIES[fam]
    ax[r, 0].imshow(demo.X[i_demo])
    ax[r, 0].set_ylabel(textwrap.fill(title, 18), fontsize=8)
    for c, (lab, param) in enumerate(levels, start=1):
        ax[r, c].imshow(h.make_variation(fam, param)(h.to_float(demo.X[i_demo]), np.random.default_rng([0, int(i_demo)])))
        ax[r, c].set_title(lab, fontsize=8)
    for a in ax[r]:
        a.set_xticks([]); a.set_yticks([])
        for sp in a.spines.values():
            sp.set_visible(False)
ax[0, 0].set_title("original", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
BATCH_CSV = Path("robustness_batch.csv")  # finished conditions are read from here, so an interrupted run continues
SCORES_DIR = Path("robustness_scores")    # the score of every image per condition, for the distribution plots
SCORES_DIR.mkdir(exist_ok=True)
CONDITIONS = [("none", "-", lambda im, rng: im)] + [
    (fam, lab, h.make_variation(fam, param)) for fam, (_, levels, _) in h.FAMILIES.items() for lab, param in levels]

done = pd.read_csv(BATCH_CSV, dtype={"level": str}) if BATCH_CSV.exists() else pd.DataFrame(columns=["family", "level", "category"])
done_keys = set(zip(done["family"], done["level"], done["category"]))
batch_rows = done.to_dict("records")

for ctg in CATEGORIES:
    todo = [c for c in CONDITIONS if (c[0], c[1], ctg) not in done_keys]
    if not todo:
        continue
    cat = h.load_category(ctg)
    ref = REFS[ctg]
    val_idx, _ = h.held_out_split(cat)
    idx = h.select(cat, "test")
    is_def = cat.labels[idx] != "good"
    store_path = SCORES_DIR / f"{ctg}.npz"
    store = dict(np.load(store_path)) if store_path.exists() else {}
    store["is_defective"], store["ref_val"] = is_def, ref.val_scores
    t0 = time.perf_counter()
    for fam, lab, fn in todo:
        s = h.score_augmented(ref, cat.X, idx, fn)
        s_val = h.score_augmented(ref, cat.X, val_idx, fn)  # held-out good images changed the same way
        row = {"family": fam, "level": lab, "category": ctg, "auroc": roc_auc_score(is_def, s),
               "good_median": float(np.median(s[~is_def]))}
        for t in TARGETS:
            row[f"caught@{t:.0%}"], row[f"frr@{t:.0%}"] = h.operating_point(ref.val_scores, s[~is_def], s[is_def], t)
            row[f"recal_caught@{t:.0%}"], row[f"recal_frr@{t:.0%}"] = h.operating_point(s_val, s[~is_def], s[is_def], t)
        batch_rows.append(row)
        pd.DataFrame(batch_rows).to_csv(BATCH_CSV, index=False)
        store[f"{fam}__{lab}__test"], store[f"{fam}__{lab}__val"] = s, s_val
        np.savez(store_path, **store)
    print(f"{ctg:11s} {len(todo)} conditions in {time.perf_counter() - t0:.0f} s", flush=True)

batch = pd.DataFrame(batch_rows)
batch.round(3).head(8)

### How big is the change?

The strengths above are nominal ("level 2 of blur" cannot be compared with "level 2 of colour"), so the change is measured on the pixels, on a sample of good test images, with the same per-image random draws as in the scoring. Two measures: the **mean absolute difference** to the original (0 to 1 scale) and the **changed area**, the share of pixels that changed by more than 0.02. The first is a common scale for global changes. It misses local changes: a few dark spots change almost no pixel on average. The second tells the two kinds apart.

In [ ]:
dose = viz.measure_dose(n_per_category=10)  # a minute or two
viz.plot_dose(dose, "mean abs difference")
viz.plot_dose(dose, "changed area")
plt.show()

In [ ]:
# what every variation does to one good image: the absolute difference to the original, amplified, one scale per row
viz.plot_differences(demo, i_demo)
plt.show()

Each variation moves the property it should (for example brightness 0.50 for the gain of 0.5, contrast 0.40 for the contrast factor 0.4, red 1.20 and blue 0.80 for the warm cast of 20%, sharpness 0.004 for the blur of 4 px); the table of all statistics is `dose.groupby(["family", "level"]).median()`. The lighting, contrast and colour changes touch most of the image, blur and noise change a share of the pixels that grows with the strength, and the dirt on the lens and the dust or oil change 1 to 2% of the pixels or less, at the same places in every image (the lens) or at random places (the dust).

### What the variations do to the results

For every condition and category (`robustness_batch.csv`): the image AUROC, and at the threshold aimed at 5% false rejects the defects caught and the real false-reject rate, both with the threshold fixed on the unchanged held-out good images and with the threshold set again on held-out good images that were changed in the same way.

In [ ]:
batch = pd.read_csv(BATCH_CSV, dtype={"level": str})
base = batch[batch.family == "none"].set_index("category")
chg = batch[batch.family != "none"].assign(auroc_change=lambda d: d.auroc - d.category.map(base.auroc))
summary = chg.groupby(["family", "level"], sort=False).agg(
    auroc_change=("auroc_change", "mean"), worst_auroc_change=("auroc_change", "min"),
    rejected_fixed=("frr@5%", "mean"), rejected_set_again=("recal_frr@5%", "mean"),
    caught_fixed=("caught@5%", "mean"), caught_set_again=("recal_caught@5%", "mean"))
print("unchanged images, mean over the six categories: AUROC %.3f, rejected %.0f%%, caught %.0f%%"
      % (base.auroc.mean(), 100 * base["frr@5%"].mean(), 100 * base["caught@5%"].mean()))
summary.round(2)

In [ ]:
# the strongest level of every variation, per category
viz.plot_matrices(batch)
plt.show()

In [ ]:
# the scores themselves: a few cases that show the two kinds of damage
# dashed line = threshold fixed on the unchanged held-out good images, tick = threshold set again on changed good images
viz.plot_scores("carpet", "brighter")     # the whole good cloud moves up, the defects stay apart: only the threshold broke
viz.plot_scores("leather", "lens_spots")  # the good cloud moves into the defects: the ranking itself is damaged
viz.plot_scores("transistor", "blur")     # a mixed case: the clouds also get closer
plt.show()

### Findings

- **The ranking is more robust than the threshold.** The mean AUROC over the six categories changes by 0.03 or less in nearly every condition. The exceptions are blur of 4 px (-0.09; transistor -0.25), dirt on the lens (-0.06 to -0.12) and four dust spots (-0.04). With the threshold fixed on the unchanged images, the share of good parts rejected (aimed at 5%, 9% on the unchanged images) grows to 57% for brighter lighting by 1.5, 92% for contrast doubled, 58% for blur of 4 px, 40% for a warm colour cast of 20%, and 41 to 60% for dirt on the lens. Even one harmless dust spot gives 25%.
- **Setting the threshold again repairs the false rejects but not always the detection.** On held-out good images changed in the same way, the false-reject rate returns to 4 to 12% in every condition. For changes of the whole image (lighting, contrast, colour) the share of defects caught hardly moves (mean 83% unchanged, 77 to 84% after the change). For local changes it falls even after the threshold is set again: dirt on the lens to 60 to 68%, dust or oil to 67 to 71%, blur of 4 px to 62%. There the scores of good and defective parts come closer together (see the leather and the transistor plots): the model is damaged and a new threshold cannot repair it.
- **Little effect:** darker lighting, uneven lighting (up to +-30% at the corners), flicker, noise up to 0.02, and lower contrast down to 0.6. The pill is hardly sensitive to colour either (AUROC 0.84 to 0.78 at the worst, false rejects unchanged at 4%).
- **The textures are the most exposed.** Carpet, grid and leather reach 60 to 100% false rejects under brighter lighting, higher contrast, blur and, for carpet and grid, the colour casts, and for dirt on the lens (71 to 100%). The objects (pill, metal nut, transistor) mostly stay below 60%, except under higher contrast (72 to 95%). A likely reason is that the scores of good textures lie in a narrow band, so a small shift crosses the threshold. This was not tested.

**Reading the numbers.** The test sets have 21 to 60 good images, so one image is 2 to 5 points of false-reject rate and differences of that size are noise; the unchanged images already overshoot the 5% target (9% on average). The variations are simulated, with strengths fixed before the run and not tuned; they are not real lot variation, and the effect of a real change in the part itself is not tested.

## Does the classical golden template cope better?

The same changes, the same split of good images, the same thresholds, now with the golden-template method of the first notebook (per-pixel mean and standard deviation of the aligned good images, ECC alignment, 512 px, grey image). The template is weak on its own: unchanged, it separates good from defective only a little on transistor, pill and metal nut, and not at all on the textures (AUROC 0.29 to 0.56). So the two methods are compared on **what a change does to each of them**, not on who is better, and the comparison is made on the three objects. The textures are shown separately, with that caveat.

One run takes about an hour (the template is not faster than the neural method at this size: 40 to 150 ms per image). Results are stored in `robustness_batch_template.csv` and `robustness_scores_template/`, and finished conditions are skipped.

In [ ]:
TEMPLATE_CSV, TEMPLATE_SCORES = Path("robustness_batch_template.csv"), Path("robustness_scores_template")
n_expected = len(h.conditions()) * len(CATEGORIES)
if TEMPLATE_CSV.exists() and len(pd.read_csv(TEMPLATE_CSV)) == n_expected:
    batch_t = pd.read_csv(TEMPLATE_CSV, dtype={"level": str})
else:
    trefs = {ctg: h.fit_template_reference(h.load_category(ctg)) for ctg in CATEGORIES}  # a few minutes
    batch_t = h.run_batch(trefs, TEMPLATE_CSV, TEMPLATE_SCORES)

# check against the first notebook (operating-point table, column "template z + ECC (512 px)"): caught (false rejects)
FIRST_T = {"transistor": ((0.50, 0.03), (0.40, 0.00)), "pill": ((0.29, 0.00), (0.21, 0.00)), "metal_nut": ((0.42, 0.09), (0.37, 0.05)),
           "leather": ((0.77, 1.00), (0.75, 1.00)), "carpet": ((0.07, 0.18), (0.00, 0.04)), "grid": ((0.00, 0.00), (0.00, 0.00))}
base_t = batch_t[batch_t.family == "none"].set_index("category")
problems = []
for ctg, ops in FIRST_T.items():
    for t, (c, f) in zip(TARGETS, ops):
        if abs(base_t.loc[ctg, f"caught@{t:.0%}"] - c) > 0.0105 or abs(base_t.loc[ctg, f"frr@{t:.0%}"] - f) > 0.0105:
            problems.append(f"{ctg} at {t:.0%}: caught {base_t.loc[ctg, f'caught@{t:.0%}']:.0%} ({base_t.loc[ctg, f'frr@{t:.0%}']:.0%}) instead of {c:.0%} ({f:.0%})")
print("the template reproduces the first study" if not problems else "differences to the first study:" + chr(10) + chr(10).join(problems))
base_t[["auroc", "caught@5%", "frr@5%"]].round(3)

In [ ]:
# the strongest level of every variation, three objects: mean over the categories, bar = lowest to highest category
viz.plot_compare({"neural network (nearest neighbour)": batch, "golden template": batch_t})
plt.show()

In [ ]:
# the template per category, as for the neural method above (textures included, where the unchanged template is already near chance)
viz.plot_matrices(batch_t)
plt.show()

In [ ]:
# the scores: darker lighting, where the template's ranking itself breaks, and dirt on the lens, where the neural method's does
viz.plot_scores("transistor", "darker", scores_dir=TEMPLATE_SCORES, method="golden template")
viz.plot_scores("transistor", "darker", method="neural network")
viz.plot_scores("leather", "lens_spots", scores_dir=TEMPLATE_SCORES, method="golden template")
viz.plot_scores("leather", "lens_spots", method="neural network")
plt.show()

### Findings

Three objects (transistor, pill, metal nut), mean over the categories, strongest level of every variation; the fixed threshold is aimed at 5% false rejects. Unchanged, the neural method catches 65% of the defects and the template 40%, so this is about what a change does to each method, not about which is better.

- **Lighting is the template's weak point; the neural method hardly moves.** Darker lighting by half: the template loses 0.29 AUROC (to chance level: 0.41 on transistor, 0.43 on metal nut) and rejects 100% of the good parts, the neural method loses 0.02 and rejects 3%. Setting the threshold again does not repair it: the template then catches 12% of the defects, the neural method 55%. Uneven lighting (up to +-30% at the corners) is harmless for the neural method (8% rejected) and breaks the template's threshold (78%). The same for flicker of +-40% (9% against 69%; AUROC -0.02 against -0.10).
- **Brighter lighting and higher contrast break both thresholds** (brighter 1.5: 39% against 100%; contrast doubled: 85% against 100%). The neural method keeps its ranking (-0.04 to -0.05) and recovers with a new threshold (catches 60%); the template's ranking also drops under contrast (-0.15), and after a new threshold it catches 19 to 51%.
- **The template does not care about colour or dirt on the lens, the neural method does a little.** A colour cast of 20% costs the template nothing (it works on a grey image; AUROC +0.02, 10% rejected) and the neural method 0.03 to 0.04. Eight dirt spots on the lens cost the template 0.01 AUROC and the neural method 0.08 (and up to 0.41 on leather, see above); with a new threshold the template still catches its 40%, the neural method 57% (from 65%).
- **Blur of 4 px hurts the two in different ways.** The neural method loses ranking (-0.13) but its threshold moves less (22% rejected); the template keeps its ranking (-0.03) but its threshold breaks (57% rejected).
- **Harmless for both on these objects:** dust or oil (the neural method suffered on leather), mild noise, colour casts of 10% or less.

**What this means.** Neither is robust to everything. The template needs the brightness of the whole image to stay put: anything that moves it, evenly or unevenly, breaks the threshold, and the stronger changes also break the ranking, so a new threshold is not enough. The neural method tolerates those, and breaks on damage that is local or removes detail (dirt on the lens, blur), where a new threshold does not help either. A line with a controlled, stable light would suit the template better than one with drift; a line with a dirty or defocused camera would suit the neural method worse.

**Reading the numbers.** Three categories, 21 to 60 good test images each, simulated changes; the template is weak even unchanged (metal nut 0.66), so its changes of AUROC are small differences of a weak signal. On the textures the unchanged template is near or below chance (AUROC 0.29 to 0.56), so its robustness there says nothing (the matrices above are shown for completeness).

## Shifted or rotated part

A part that sits a little off in the fixture. The test images (and the held-out good images for the new threshold) are **shifted** by 4, 12 and 32 px (about 1%, 2.5% and 6% of the width) in a random direction, or **rotated** by 1, 3 and 8 degrees to the left or the right, with the border mirrored. Same fixed models and thresholds as before, for both methods. The template aligns every image to its template first (ECC, rotation and translation), so it should cope with moderate shifts and rotations by design. A shift or a rotation uncovers a strip or corners of the image; the simulation fills them with a mirrored copy of the part. That turns out to matter, and is tested below the plots.

In [ ]:
GEO_FILES = {"neural network (nearest neighbour)": ("robustness_batch_geometry.csv", "robustness_scores_geometry"),
             "golden template": ("robustness_batch_geometry_template.csv", "robustness_scores_geometry_template")}
n_geo = len(h.conditions(h.GEOMETRY)) * len(CATEGORIES)
geo = {}
for name, (csv, scores_dir) in GEO_FILES.items():
    if Path(csv).exists() and len(pd.read_csv(csv)) == n_geo:
        geo[name] = pd.read_csv(csv, dtype={"level": str})
    else:  # about 15 to 30 minutes per method
        fit = h.fit_reference if name.startswith("neural") else h.fit_template_reference
        geo[name] = h.run_batch({c: fit(h.load_category(c)) for c in CATEGORIES}, csv, scores_dir, families=h.GEOMETRY)
geo_nn, geo_tm = geo["neural network (nearest neighbour)"], geo["golden template"]

# a transistor at the three strengths of both changes (the unchanged image is the first column)
tr = h.load_category("transistor")
i_tr = h.select(tr, "test", good=True)[0]
fig, ax = plt.subplots(2, 4, figsize=(9, 4.6))
for r, fam in enumerate(h.GEOMETRY):
    title, levels, _ = h.GEOMETRY[fam]
    ax[r, 0].imshow(tr.X[i_tr]); ax[r, 0].set_title("original", fontsize=8); ax[r, 0].set_ylabel(textwrap.fill(title, 18), fontsize=8)
    for c, (lab, param) in enumerate(levels, start=1):
        ax[r, c].imshow(h.make_variation(fam, param)(h.to_float(tr.X[i_tr]), np.random.default_rng([0, int(i_tr)])))
        ax[r, c].set_title(lab, fontsize=8)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# the strongest level, mean over the three objects (bar = lowest to highest category) and over the three textures
viz.plot_compare(geo, families=h.GEOMETRY)
plt.title("three objects", loc="left", fontsize=9)
viz.plot_compare(geo, categories=("leather", "carpet", "grid"), families=h.GEOMETRY)
plt.title("three textures (the unchanged template is near chance there)", loc="left", fontsize=9)
plt.show()

In [ ]:
for name, b in geo.items():
    print(name)
    viz.plot_matrices(b, families=h.GEOMETRY)
    plt.show()
viz.plot_scores("transistor", "rotate", scores_dir="robustness_scores_geometry", method="neural network")
viz.plot_scores("transistor", "rotate", scores_dir="robustness_scores_geometry_template", method="golden template")
viz.plot_scores("metal_nut", "shift", scores_dir="robustness_scores_geometry_template", method="golden template")
plt.show()

### Is it the shift, or the strip it uncovers?

The mirrored strip (or the mirrored corners, for a rotation) is not part of a real shift. To see how much of the damage it causes, the unchanged, the shifted (32 px) and the rotated (8 degrees) test images are scored again with a **frame around the image ignored**, with the threshold set on the unchanged held-out good images scored the same way. The template ignores a frame of 40 and 60 px, the neural method 3 and 4 patches (54 and 73 px); "own" is the margin the neural method chose for itself in the first study (0 for these objects).

In [ ]:
frame = sc.load()  # about 10 minutes the first time, then read from shift_frame_check.csv
for method, d in frame.groupby("method"):
    print(method + ": image AUROC and share of good parts rejected")
    display(d.pivot_table(index=["category", "frame ignored"], columns="condition", values=["auroc", "good rejected"], sort=False)
              .reindex(columns=pd.MultiIndex.from_product([["auroc", "good rejected"], ["unchanged", "shift 32 px", "rotation 8 degrees"]])).round(2))

### Findings

Three objects, mean over the categories (transistor, pill, metal nut); fixed threshold aimed at 5% false rejects.

- **The template copes with rotation, the neural method partly does not (on an oriented part).** At 8 degrees the template keeps its AUROC (+0.01) and rejects 9% of the good parts; the neural method loses 0.09 (transistor 0.26) and rejects 30% (transistor 90%). Most of the transistor's loss is the mirrored corners of the simulation: with a frame of 3 patches ignored the neural AUROC is 0.81 (unchanged 0.88) and 28% are rejected, against 0.61 and 90%. A loss of about 0.07 remains, which the template does not have. Pill and metal nut hardly care.
- **A shift mostly costs the strip it uncovers, not the detection.** Up to 12 px (2.5% of the width) both methods change little (AUROC -0.04 or less, 12% (template) to 15% (neural) rejected). At 32 px (6%) both seemed to lose 0.20 AUROC and to reject 52% (neural) and 64% (template) of the good parts. With a frame as wide as the shift ignored, the neural method is back at its unchanged AUROC on all three objects (transistor 0.57 to 0.88 and 70% to 13% rejected; metal nut 0.71 to 0.96 and 68% to 9%) and so is the template on pill (0.57 to 0.73, 54% to 0%) and metal nut (0.35 to 0.63, 91% to 9%). Only the template on the transistor stays low (0.69, 38% rejected): its alignment diverged on 3 of 12 sampled transistor images (and 2 of 12 metal nut images), and a shifted transistor may look like its "misplaced" defect type.
- **On the textures the neural method keeps its ranking** (AUROC change 0.01 or less) but its fixed threshold drifts: 47% of the good parts rejected at 32 px and 45% at 8 degrees (grid 86%), against 10 to 13% when set again, with 89 to 99% of the defects still caught. This is probably the same strip effect; it was not tested on the textures. The unchanged template is near chance on the textures, so its numbers there say nothing.
- **For a line:** rotation of a part is better handled by the template with its alignment. A shift costs the strip of the image it uncovers; ignoring a frame as wide as the shift is enough in these tests, at the price of not seeing defects in that frame. In a real setup that strip shows background, not a mirrored part.

**Reading the numbers.** Same caveats as before: three categories, 21 to 60 good test images, simulated changes with mirrored borders (which turned out to matter, see above; a shift or rotation of a real part would also move the background and the lighting).

### Can combining the two help?

The two methods break under different changes, so one might combine them. The saved scores of both are scaled with the mean and standard deviation of the held-out good images of their own method, and combined image by image: their mean, their maximum (reject if either says so) and their minimum (reject only if both say so). The threshold is fixed on the held-out good images, as before. No new run is needed.

In [ ]:
fused = viz.fusion_table()
objs = fused[fused.category.isin(["transistor", "pill", "metal_nut"])]
rules = ["neural", "template", "mean", "max", "min"]
unchanged = objs[objs.family == "none"].groupby("rule")[["auroc", "frr", "caught"]].mean().loc[rules]
print("unchanged images, mean over the three objects")
display(unchanged.round(2))

strongest = {f: v[1][-1][0] for f, v in {**h.FAMILIES, **h.GEOMETRY}.items()}
is_strongest = np.array([strongest.get(f) == l for f, l in zip(objs.family, objs.level)])
ch = objs[(objs.family != "none").values & is_strongest]
base_auroc = objs[objs.family == "none"].set_index(["category", "rule"]).auroc
ch = ch.assign(auroc_change=ch.auroc.values - base_auroc.loc[list(zip(ch.category, ch.rule))].values)
per_family = ch.groupby(["rule", "family"]).agg(rejected=("frr", "mean"), auroc_change=("auroc_change", "mean"))
print("over the 14 variations at their strongest level: mean and worst case of the share of good parts rejected, worst AUROC change")
display(per_family.groupby("rule").agg(mean_rejected=("rejected", "mean"), worst_rejected=("rejected", "max"), worst_auroc_change=("auroc_change", "min")).loc[rules].round(2))
display(ch.pivot_table(index="family", columns="rule", values="frr", sort=False)[rules].round(2))

Simple fusion does not give a more robust method. The mean and the maximum are a little better unchanged (AUROC 0.92 against 0.89 for the neural method and 0.74 for the template), but under a change they inherit the fragile method's failure: the template's scores grow so much under darker lighting or higher contrast that the fused score rejects 100% of the good parts, and under flicker the mean loses more AUROC (-0.17) than either method alone. The minimum rule, which rejects only when both agree, stays calm under lighting (11% rejected for darker lighting, 24% for flicker) and loses at most 0.10 AUROC, but it gives up detection (unchanged AUROC 0.79, 49% caught against 65%) and still breaks under the simulated shift (which is mostly the mirrored strip, see above). Averaged over the 14 variations the neural method alone rejects the fewest good parts (23%), then the minimum rule (37%), the template (55%), the maximum (58%) and the mean (59%). A better combination would have to use the knowledge of which change happened (for example, a lighting check that tells the template to trust or not), not just the two scores.

### Something more robust?

The failures found so far point at two cheap changes, tested here on the three objects at the strongest level of every variation (the 14 changes of lighting, contrast, colour, camera, lens, shift and rotation):

1. **The template with a brightness normalisation.** After the alignment, every image is fitted to the mean image of the good ones, `image = a * template + b + c * x + d * y` (gain, offset and a linear shading), and the fit is removed before the z-map. This targets the template's weak point, any change of brightness. The same is done for the good training images, so the per-pixel spread no longer contains lighting differences.
2. **The neural method on aligned images.** Every image, in training too, is aligned to the template first (ECC, rotation and translation) and the features are taken from the aligned image. This targets the neural method's weak points, rotation and shift.

A third idea, starting the alignment from a shift found by phase correlation (for large shifts), was tried on the transistor only and dropped: it made the alignment worse on unchanged images (AUROC 0.84 down to 0.65).

Each run takes about 15 to 25 minutes per method; the results are stored in `robustness_batch_template_photo.csv` and `robustness_batch_neural_aligned.csv` (and the `robustness_scores_*` folders) and are read from there if present.

In [ ]:
OBJECTS = ["transistor", "pill", "metal_nut"]
ALL_FAMILIES = {**h.FAMILIES, **h.GEOMETRY}
CAND = {"template_photo": h.fit_normalised_template,
        "neural_aligned": lambda cat: h.fit_aligned_reference(cat, phase_init=False)}
cand = {}
for name, fit in CAND.items():
    csv = Path(f"robustness_batch_{name}.csv")
    if csv.exists() and len(pd.read_csv(csv)) == len(h.conditions(ALL_FAMILIES, True)) * len(OBJECTS):
        cand[name] = pd.read_csv(csv, dtype={"level": str})
    else:  # about 15 to 25 minutes per method
        refs = {c: fit(h.load_category(c)) for c in OBJECTS}
        cand[name] = h.run_batch(refs, csv, f"robustness_scores_{name}", families=ALL_FAMILIES, strongest_only=True)

def both(*files):  # the lighting and the geometry runs of one method, as one table
    return pd.concat([pd.read_csv(f, dtype={"level": str}) for f in files]).drop_duplicates(["family", "level", "category"])

methods = {"neural network (nearest neighbour)": both("robustness_batch.csv", "robustness_batch_geometry.csv"),
           "golden template": both("robustness_batch_template.csv", "robustness_batch_geometry_template.csv"),
           "template with brightness normalisation": cand["template_photo"],
           "neural network on aligned images": cand["neural_aligned"]}

viz.plot_compare({k: v for k, v in list(methods.items())[:3]}, families=ALL_FAMILIES)
plt.show()

In [ ]:
# the four methods over the 14 variations at their strongest level (three objects)
strongest = {f: v[1][-1][0] for f, v in ALL_FAMILIES.items()}
rows, unchanged = {}, {}
for name, d in methods.items():
    d = d[d.category.isin(OBJECTS)]
    b = d[d.family == "none"].set_index("category")
    x = d[np.array([strongest.get(f) == l for f, l in zip(d.family, d.level)]) & (d.family != "none").values].copy()
    x["auroc_change"] = x.auroc - x.category.map(b.auroc)
    g = x.groupby("family").agg(rejected=("frr@5%", "mean"), auroc_change=("auroc_change", "mean"), caught_set_again=("recal_caught@5%", "mean"))
    unchanged[name] = b[["auroc", "caught@5%", "frr@5%"]].mean()
    rows[name] = {"mean rejected": g.rejected.mean(), "worst rejected": g.rejected.max(), "mean AUROC change": g.auroc_change.mean(),
                  "worst AUROC change": g.auroc_change.min(), "mean caught, threshold set again": g.caught_set_again.mean()}
print("unchanged images, mean over the three objects")
display(pd.DataFrame(unchanged).T.round(2))
print("over the 14 variations")
display(pd.DataFrame(rows).T.round(2))

### Findings

Three objects, fixed threshold aimed at 5% false rejects.

- **Normalising the brightness fixes the template's weak point.** For darker and brighter lighting, lower and higher contrast and flicker the share of good parts rejected falls from 69 to 100% to 4 to 6%, and the AUROC no longer changes (+-0.02); uneven lighting falls from 78% to 21% and noise from 34% to 6%. Unchanged it is also a little better than the plain template (AUROC 0.77 against 0.74, 45% of the defects caught against 40%). Averaged over the 14 variations it rejects the fewest good parts of the four methods (17%, against 23% for the neural method and 55% for the plain template); without the shift the order is the same (13%, against 21% and 54%).
- **It is robust, but not sensitive.** It still catches fewer defects than the neural method (45% against 65% unchanged, and 40% against 56% on average over the variations after the threshold is set again). A better threshold stability bought with a weaker detection is the trade.
- **It does not fix what it was not built for:** blur (-0.12 AUROC, 40% rejected) and dirt on the lens (37%). A shift of 32 px gives -0.18 and 68% rejected, but most of that is the mirrored strip (see above).
- **Aligning the images before the neural method helps little.** Unchanged it is the same (AUROC 0.90, 61% caught), rotation improves (false rejects 30% to 17%, AUROC change -0.09 to -0.06), and the mean share of good parts rejected is 22% against 23% over the 14 variations (19% against 21% without the shift): within the noise of these test sets.
- **So:** the most robust single change tested is a preprocessing step that removes a known failure, not a different model, and it comes at the price of detection. For blur and dirt on the lens none of the tested changes helps; those need a different remedy (a clean and focused camera, a fixture, or other features, which were not tested).

**Reading the numbers.** Three categories, 21 to 60 good test images each, simulated changes, one threshold target; differences of a few points are noise.

### Another pretrained model?

The neural method so far used the features of one network (Wide ResNet50, trained on ImageNet labels). Are its failures a property of the method, or of those features? The same method (memory of good patches, nearest neighbour, border margin from held-out good images, same split and threshold) is run with the features of two other pretrained networks:

- **ConvNeXt-tiny** (also trained on ImageNet labels), stage 3 (14 x 14 grid, 384 channels), the stage that was best in the first study;
- **DINOv2 ViT-S/14** (self-supervised, trained with strong colour and blur augmentation), 336 px input, a 24 x 24 grid of patch features.

Three objects, strongest level of every change, as for the other candidates. One run takes about 15 minutes (ConvNeXt) and 30 to 40 minutes (DINOv2); the results are stored in `robustness_batch_convnext.csv` and `robustness_batch_dinov2.csv`.

In [ ]:
nets = {}
for name, make in {"convnext": lambda: h.ConvNeXtNet(stage=5), "dinov2": lambda: h.DinoNet(336)}.items():
    csv = Path(f"robustness_batch_{name}.csv")
    if csv.exists() and len(pd.read_csv(csv)) == len(h.conditions(ALL_FAMILIES, True)) * len(OBJECTS):
        nets[name] = pd.read_csv(csv, dtype={"level": str})
    else:  # about 15 to 40 minutes
        net = make()
        refs = {c: h.fit_net_reference(h.load_category(c), net) for c in OBJECTS}
        nets[name] = h.run_batch(refs, csv, f"robustness_scores_{name}", families=ALL_FAMILIES, strongest_only=True)

backbones = {"neural network (Wide ResNet50)": methods["neural network (nearest neighbour)"],
             "ConvNeXt-tiny": nets["convnext"], "DINOv2 small": nets["dinov2"]}
viz.plot_compare(backbones, families=ALL_FAMILIES)
plt.show()

# all five methods, over the 13 changes without the shift (its large values are mostly the mirrored strip, see above)
unchanged, over = viz.summary_table({"golden template": methods["golden template"],
                                     "template with brightness normalisation": methods["template with brightness normalisation"],
                                     **backbones}, ALL_FAMILIES)
print("unchanged images, mean over the three objects")
display(unchanged.round(2))
print("over the 13 changes at their strongest level (without the shift)")
display(over.round(2))

### Findings

Three objects, fixed threshold aimed at 5% false rejects, the 13 changes without the shift.

- **Unchanged, DINOv2 is clearly the best detector** (AUROC 0.97, 80% of the defects caught, against 0.89 and 65% for the Wide ResNet50 and 0.93 and 66% for ConvNeXt), and after a new threshold it still catches the most (71% on average over the changes, against 59% and 60%).
- **The failure profile follows the features, not the method.** DINOv2 hardly notices lighting, contrast, colour, noise, flicker or rotation: darker, brighter, lower contrast, uneven lighting and flicker reject 4 to 9% of the good parts (AUROC change +0.01 to -0.01), rotation 9%, and blur costs -0.06 AUROC and 29% rejected (the Wide ResNet50: -0.13 and 22%). Its weak point is the opposite: dirt on the lens and harmless dust, which it reads as anomalies, reject 92% and 69% of the good parts (AUROC -0.21 and -0.18), and higher contrast 42%. ConvNeXt is the reverse in another place: blur breaks it (AUROC -0.34, 79% rejected, 11% of the defects caught after a new threshold) and so, less, do dust (-0.10, 23%) and noise (-0.07, 21%), while dirt on the lens (7%) and most lighting changes (4 to 21% rejected) are fine; higher contrast rejects 45%. The probable reason for DINOv2's profile is that it was trained to ignore the changes of colour, blur and crop that it ignores here, and to see spots as content; this was not tested.
- **No pretrained network is robust to everything.** Averaged over the 13 changes the share of good parts rejected is the same for the three (19 to 22%, template with brightness normalisation 13%), and each has a change that rejects 79 to 92% of the good parts (the worst cases: Wide ResNet50 85% for doubled contrast, ConvNeXt 79% for blur, DINOv2 92% for dirt on the lens). The first study's choice of method therefore says less about robustness than the choice of features and the kind of change a line sees: clean optics suit DINOv2 best (accuracy and robustness), a dirty lens or dust suit it worst.
- **The shift is left out** because its large values are mostly the simulation's mirrored strip; the three networks lose 0.08 to 0.20 AUROC at 32 px, which cannot be read as an effect of the shift.

**Reading the numbers.** Three categories, 21 to 60 good test images each, simulated changes, one setting per network (ConvNeXt stage 3, DINOv2 small at 336 px); other networks, sizes and stages, and PatchCore, were not tested.

## Threshold flow with a few reject samples

So far the threshold came from good parts alone. On a line there is another way: collect a few known rejects, **tune the threshold so that they are caught without rejecting too many good parts, and check it on parts that were not used**. How many reject samples does that take, and does it help when something has changed?

**What is simulated.** The reject samples are `k` defective test images (in practice they would be collected rejects). The good parts are the held-out 20% of the good training images, changed in the same way as the test images, as in "set again" above. The threshold is tuned to the lowest cost on those good parts and the `k` samples: `c` x (share of the samples missed) + (share of the good parts rejected), where `c` says how much worse an escaped defect is than a false reject (c = 1: equal, c = 5: five times worse; not known, so both are shown). The result is checked on everything else: the good test images and the defective test images that were not drawn as samples. 200 random draws per number of samples, mean over the six categories. Compared with the threshold fixed on the unchanged good parts and the one set again on changed good parts, neither of which uses samples.

Two cases for the samples: drawn from **all defect types** ("seen types"), or all from **one defect type**, checked on the defects of the other types only: the situation of a defect nobody has seen yet. The neural method is used, in four conditions: unchanged, and the strongest brighter lighting, dirt on the lens and blur, where the threshold drifts or the model is damaged.

In [ ]:
FLOW_CONDITIONS = [("none", "-"), ("brighter", "1.5"), ("lens_spots", "8"), ("blur", "4")]
flow = pd.concat([tf.simulate(ctg, cond) for ctg in CATEGORIES for cond in FLOW_CONDITIONS])  # a few seconds, from the saved scores

tf.plot_flow(flow, c=5)
plt.show()

In [ ]:
# the same as numbers: 10 reject samples, mean over the six categories and 200 draws
f = flow.assign(condition=lambda d: d.family.replace({"none": "unchanged"}) + " " + d.level.where(d.family != "none", ""))
t10 = f[(f.k == 10) & (f.c == 5)].groupby(["condition", "scenario", "rule"])[["caught", "rejected", "cost"]].mean()
print("10 reject samples, c = 5: share of defects caught, share of good parts rejected, cost")
display(t10.round(2).unstack("rule").reindex(columns=["fixed", "set again", "tuned"], level=1))

for c in (1, 5):
    g = f[(f.c == c) & (f.scenario == "seen types")].groupby(["condition", "rule", "k"]).cost.mean().unstack("k")
    print(f"cost with c = {c}, samples from all defect types, by the number of samples (the two threshold rules without samples do not depend on it)")
    display(g.loc[(slice(None), ["set again", "tuned"]), :].round(2))

### Findings

Neural method, six categories, mean over 200 draws of the samples.

- **A few samples help if an escaped defect is much worse than a false reject (c = 5).** Unchanged, the cost falls from 0.96 (the threshold from good parts alone) to 0.77 with 2 samples, 0.59 with 5, 0.50 with 10 and 0.47 with 20; one sample does nothing (0.94). Five to ten samples give most of the gain. If an escape costs the same as a false reject (c = 1), tuning hardly matters: 0.26 falls to 0.22, and under the changes it stays the same (blur 0.42, dirt on the lens 0.50) or improves a little (brighter lighting 0.33 to 0.28).
- **The gain is larger when something has changed.** With the threshold set again on changed good parts the cost is 1.20 (brighter lighting), 2.11 (dirt on the lens) and 1.95 (blur); with 10 samples it falls to 0.63, 0.91 and 0.90. The samples show where the defects lie in the changed scores, and the threshold moves down to catch them.
- **The price is overkill.** With 10 samples the tuned threshold catches 91 to 95% of the defects (59 to 83% for the threshold set again) but rejects 24% of the good parts unchanged (9% set again), 29% for brighter lighting, 43% for blur and 49% for dirt on the lens. Tuning moves along the same trade-off; it does not repair a damaged ranking: with dirt on the lens the cost stays at 0.91, against 0.50 unchanged. Whether a line can live with 24 to 49% overkill depends on the real value of `c`.
- **Samples from one defect type give less.** Unchanged, 10 samples cost 0.66 instead of 0.50, about two thirds of the gain; under the changes 0.79 to 1.13 instead of 0.63 to 0.91. A set of rejects that does not cover the defect types the line will see is less useful, and the check on other types shows it.
- **With c = 5 the old fixed threshold can cost less than the one set again** (brighter lighting 0.98 against 1.20): it rejects many good parts but misses fewer defects. This is the same trade-off at another point, not a better method.

**Reading the numbers.** The samples are test defects drawn at random; a real set of rejects is rarer and less representative. The tuning uses 42 to 56 good parts, and the check 21 to 60 good test images per category. The rule is one simple choice (no limit on overkill, no smoothing); `c` is a guess. The conditions are simulated.

## Write-up

### Summary

The first study asked which method separates good from defective parts best on clean images. A line does not stay clean: the light drifts, a lens gets dirty, a part sits a little off. So this study asks what survives such changes. I changed the test images in 14 simulated ways, three strengths each (lighting, contrast, colour, blur and noise, dirt on the lens, dust, shift, rotation), and scored them with models and thresholds that were fixed beforehand.

- **The ranking of good and defective parts survives most changes much better than the threshold does.** For the neural method, setting the threshold again on good parts that were changed in the same way brings the false rejects back to 4 to 12% in every condition. It does not bring back the defects when the change is local (dirt on the lens, blur): there the scores themselves are damaged.
- **The golden template and the neural method fail in opposite places.** The template fails on any change of brightness, the neural method on blur, dirt on the lens and a rotated part.
- **A small preprocessing step, normalising the brightness, made the template the most stable method tried** (17% of the good parts rejected on average over the 14 changes, against 23% for the neural method and 55% for the plain template), at the price of finding fewer defects.
- **Another pretrained network changes the failure profile but does not remove it.** DINOv2 ignores lighting, contrast and rotation but reads dirt on the lens and dust as defects (92% and 69% of the good parts rejected); ConvNeXt breaks on blur.
- **A handful of reject samples (5 to 10) can retune a threshold** if an escaped defect is much worse than a false reject, at the price of rejecting many more good parts.

Built with Claude as a coding assistant; the notebook is the full record.

### Question and approach

Can a method that was fitted on good images keep working when the images change, and what does it cost to keep it working? Models and thresholds are fixed on good training images only (80% of them fit the model, 20% are held out for the threshold, aimed at 5% false rejects). Then the **test images are changed** and re-scored with the unchanged model. Three things are read off: the image AUROC (does the ranking survive?), the defects caught and the good parts rejected at the old threshold (does the threshold survive?), and the same two numbers with a threshold set again on held-out good parts changed in the same way. Methods: a nearest-neighbour method on Wide ResNet50 features with a border margin ("neural method") and the aligned golden template of the first study. The comparison between the two is made on the three objects (transistor, pill, metal nut); on the textures the unchanged template is near chance, so its numbers there say nothing.

### What I found

1. **The threshold breaks before the ranking.** For the neural method the mean AUROC over the six categories changes by 0.03 or less in nearly every condition, but the share of good parts rejected at the fixed threshold (9% unchanged) reaches 57% for brighter lighting, 92% for doubled contrast, 58% for blur and 41 to 60% for dirt on the lens. A threshold set again repairs the false rejects and, for changes of the whole image, the share of defects caught (77 to 84% against 83% unchanged); for local changes the share caught falls to 60 to 71%.
2. **The template needs a stable light, the neural method a clean, sharp image.** Darker lighting by half costs the template 0.29 AUROC (to chance level) and rejects 100% of the good parts; the neural method loses 0.02 and rejects 3%. Uneven lighting (78% against 8% rejected) and flicker (69% against 9%) break the template's threshold too. The template ignores colour casts and dirt on the lens; the neural method loses 0.08 AUROC to the dirt and 0.13 to blur of 4 px.
3. **Rotation favours the template; a shift mostly costs the strip it uncovers.** At 8 degrees the template keeps its AUROC and rejects 9% of the good parts; the neural method loses 0.09 (transistor 0.26) and rejects 30%, but most of the transistor's loss is the mirrored corners of the simulation (with a frame ignored the loss is 0.07 and 28% are rejected). Shifts up to 2.5% of the width are tolerated by both. A shift of 6% looked like a loss of 0.20 AUROC for both, but ignoring a frame as wide as the shift brings the neural method (and the template on pill and metal nut) back to the unchanged results: the damage was the strip the shift uncovers, which the simulation filled with a mirrored copy of the part, not the shift itself.
4. **Preprocessing beat a better model, up to a point.** Fitting every image to the template's brightness, contrast and a linear shading takes the template's false rejects under lighting changes from 69 to 100% down to 4 to 6%. It still catches fewer defects than the neural method (45% against 65% unchanged) and does not help against blur or dirt on the lens. Aligning the images before the neural method helped little, a start of the alignment from phase correlation made it worse, and combining the two methods by their scores either inherits the weaker method's failures (mean, maximum) or loses detection (minimum).
5. **A different pretrained network changes the failure profile.** DINOv2 (self-supervised) is the best detector unchanged (AUROC 0.97, 80% caught against 0.89 and 65%) and hardly notices lighting, contrast, colour, noise or rotation (4 to 9% rejected), but dirt on the lens and dust reject 92% and 69% of the good parts; ConvNeXt breaks on blur (AUROC -0.34, 79% rejected). Averaged over the changes the three networks reject the same share of good parts (19 to 22%); the worst case of each is 79 to 92%.
6. **Reject samples can retune a threshold, with a price.** With an escaped defect five times as bad as a false reject, five to ten samples cut the cost from 0.96 to 0.59 and 0.50 on unchanged images, and from 1.2 to 2.1 down to 0.6 to 0.9 after a change. The tuned threshold then rejects 24% of the good parts unchanged and 29 to 49% after a change (a threshold set again rejects 9% unchanged). If all samples are of one defect type, about two thirds of the gain remains on the other types.

### What this means on a real line

- **Know which failure you have.** A drift of the whole image (light, contrast, colour) moves the scores of the good parts and breaks the threshold, but a threshold set again on good parts repairs it. Damage that is local or removes detail (a dirty or blurred lens) cannot be repaired by a threshold; it needs cleaning, focus or retraining. In these runs, when the median score of the good parts moved by more than 10% the false-reject rate rose by 0.73 on average for the neural method (0.42 for the template), and by 0.05 (0.01) when it moved less. A cheap monitor of that median would have shown the changes that break the threshold (the same images, so this shows what such a monitor sees, not an independent test).
- **Lighting is the template's job to control, focus and cleanliness the neural method's.** Where the light cannot be held, a normalisation step is cheap. Where the part can turn, the template with its alignment is safer. A shift costs the strip of the image it uncovers: ignoring a frame as wide as the shift restored the results here (and hides defects in that frame).
- **Collect rejects of different types and look at the overkill.** A few samples are worth having if a missed defect is expensive, but the threshold they produce is a trade-off that must be judged on the good parts it rejects.
- **Benchmarks like this one do not say how often these changes happen.** Which of them matters depends on the line.

### Limitations

- Six of the fifteen categories, and only three objects in the comparison between methods; 21 to 60 good test images per category, so one image is 2 to 5 points of false-reject rate and differences of that size are noise. One seed, one split, one machine.
- The changes are **simulated** (strengths fixed in advance, not tuned, and borders mirrored for shift and rotation, which caused most of the damage of a large shift and of the transistor's rotation); real lots, real lamps and real fixtures differ, and changes of the part itself, of scale or of several parts in one image are not tested.
- One threshold target (5%), and a cost ratio between a missed defect and a false reject that is a guess; the tuning rule with samples is one simple choice and the samples are test defects, which are more representative than real rejects would be.
- The template is weak even unchanged (metal nut AUROC 0.66, textures near chance), so its changes are changes of a weak signal; PatchCore and other methods were not tested under the changes (only PatchCore's scores were cached, not fitted models), of the neural family only three feature networks were (Wide ResNet50, ConvNeXt-tiny, DINOv2-small, one setting each), and the candidates for a more robust method were tested on the three objects at the strongest level of each change only.
- The phase-correlation start of the alignment was tried on one category only. The template's alignment diverged on 3 of 12 sampled transistor and 2 of 12 metal nut images at a shift of 6%; the effect of the mirrored strip was tested on the three objects only, not on the textures.

### Reproduce

The first cell holds all the imports. Set `MVTEC_AD_ROOT` (or the default in `helpers.data_root()`), then run from the top. Rough run times on this machine (CPU only): baseline 9 minutes; the batch of 36 lighting, contrast, colour, camera and lens conditions about an hour for each method; shift and rotation about 30 minutes for both methods; the two candidates about 50 minutes; the two other networks about 15 and 30 to 40 minutes; the dose measurement 1 to 2 minutes; the threshold flow and the fusion a few seconds. All results are stored in `robustness_batch*.csv` and `robustness_scores*/` and are read from there when they exist, so a rerun takes minutes; delete a file to repeat that part.